# 1. Exploratory Data Analysis

Visual and statistical exploration of the watch dataset before model development.

In [6]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

from src.utils import load_config
from src.utils.visualization import set_style, plot_price_distribution, plot_brand_analysis
from src.data import prepare_metadata, create_splits, get_transforms

set_style()
config = load_config('../configs/base.yaml')
config['data']['metadata_path'] = '../data/metadata.csv'
config['data']['root_dir'] = '../data/raw'

ModuleNotFoundError: No module named 'sklearn'

## 1.1 Dataset overview

In [ ]:
df = prepare_metadata(config)
print(f'Dataset: {len(df)} images, {df["brand"].nunique()} brands')
print(f'Price range: ${df["price_clean"].min():.0f} – ${df["price_clean"].max():.0f}')
df.head(10)

## 1.2 Price distribution

In [ ]:
plot_price_distribution(df['price_clean'].values)

## 1.3 Brand analysis

In [ ]:
plot_brand_analysis(df)

## 1.4 Image samples

Visual inspection of images across price ranges.

In [ ]:
# Sample images from different price bins
img_dir = Path(config['data']['root_dir'])
bins = {'Budget (<$100)': df[df['price_clean'] < 100],
        'Mid ($100-500)': df[(df['price_clean'] >= 100) & (df['price_clean'] < 500)],
        'Premium ($500-1K)': df[(df['price_clean'] >= 500) & (df['price_clean'] < 1000)],
        'Luxury ($1K+)': df[df['price_clean'] >= 1000]}

fig, axes = plt.subplots(4, 5, figsize=(16, 14))
for row, (label, subset) in enumerate(bins.items()):
    samples = subset.sample(min(5, len(subset)), random_state=42)
    for col, (_, s) in enumerate(samples.iterrows()):
        img = Image.open(img_dir / s['image_name'])
        axes[row, col].imshow(img)
        axes[row, col].set_title(f'${s["price_clean"]:.0f}', fontsize=9)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=0, ha='right', va='center')
plt.suptitle('Sample images by price range', fontsize=14)
plt.tight_layout()

## 1.5 Image properties

In [ ]:
# Check image sizes, aspect ratios, color stats
sizes = []
for fname in df['image_name'].sample(200, random_state=42):
    img = Image.open(img_dir / fname)
    sizes.append(img.size)

widths, heights = zip(*sizes)
print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')

## 1.6 Train / Val / Test split

In [ ]:
train_df, val_df, test_df = create_splits(df, config)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, split) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    ax.hist(np.log1p(split['price_clean']), bins=30, alpha=0.7)
    ax.set_title(f'{name} (n={len(split)})')
    ax.set_xlabel('log(1 + price)')
plt.suptitle('Price distribution per split (log scale)', fontsize=13)
plt.tight_layout()

## 1.7 Augmentation preview

In [ ]:
# Show same image with different augmentations
sample_img = np.array(Image.open(img_dir / df.iloc[0]['image_name']).convert('RGB'))
transform = get_transforms(config, 'train')

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original')
for ax in axes.flat[1:]:
    aug = transform(image=sample_img)['image']
    # Denormalize for display
    aug_np = aug.numpy().transpose(1, 2, 0)
    aug_np = (aug_np * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]).clip(0, 1)
    ax.imshow(aug_np)
for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Augmentation samples', fontsize=13)
plt.tight_layout()